# EDM3: Notebook A - Trajectory Generator (FAST / GPU)

**Instructions:**
- Set **Accelerator: GPU T4 x2 (or P100)**.
- Run all cells.
- Output `unscored_trajectories.npz` will be saved to `/kaggle/working/`.
- Export the `/kaggle/working` directory as a new Kaggle Dataset named `unscored-trajectories`.

In [ ]:
import os, sys
import subprocess

print("Installing JAX with GPU support...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]", "flax", "optax"], check=True)
print("Dependencies installed.")

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import flax.linen as nn
from jax import tree_util
import time
from tqdm.notebook import tqdm

# ── Global Config ──────────────────────────────────────────────
SEQ_LEN         = 100_000       # MDP sequence length
VOCAB_SIZE      = 5             # A, C, G, T, N
NUM_EDITS       = 10            # Mutations per trajectory
METADATA_DIM    = 10            # Demographic metadata dimension
TEMPERATURE     = 2.0           # Sampling temperature for exploration
NUM_TRAJECTORIES = 5000         # Generation count (increased for Phase 6)

DATA_DIR = "/kaggle/working"
os.makedirs(DATA_DIR, exist_ok=True)
TRAJ_PATH = os.path.join(DATA_DIR, "unscored_trajectories.npz")

print(f"JAX devices: {jax.devices()}")

In [ ]:
class GeneratorPolicyV2(nn.Module):
    """
    SOTA Dual-Head GFlowNet Policy:
      - Shared Conv1D backbone (aggressive striding)
      - Factored action head (position + base scores)
      - Value head V(s) for Sub-EB
    """
    seq_len: int = SEQ_LEN
    vocab_size: int = VOCAB_SIZE
    input_channels: int = 6  # 5 one-hot + mutated mask

    @nn.compact
    def __call__(self, state_input, target_metadata):
        B = state_input.shape[0]
        L = self.seq_len

        # Shared Conv1D backbone
        x = nn.Conv(features=16, kernel_size=(10,), strides=(10,))(state_input)
        x = jax.nn.relu(x)
        x = nn.Conv(features=32, kernel_size=(10,), strides=(10,))(x)
        x = jax.nn.relu(x)  # (B, L/100, 32)

        # Position scores
        pos_coarse = nn.Conv(features=1, kernel_size=(1,), strides=(1,))(x).squeeze(-1)
        pos_scores = jnp.repeat(pos_coarse, 100, axis=-1)[:, :L]

        # Global features
        global_feat = jnp.concatenate([jnp.mean(x, axis=1), target_metadata], axis=-1)
        g = jax.nn.relu(nn.Dense(128)(global_feat))
        g = jax.nn.relu(nn.Dense(64)(g))

        # Action head
        base_scores = nn.Dense(self.vocab_size)(g)
        terminate_logit = nn.Dense(1)(g)
        combined = pos_scores[:, :, None] + base_scores[:, None, :]
        mutation_logits = combined.reshape((B, L * self.vocab_size))
        action_logits = jnp.concatenate([mutation_logits, terminate_logit], axis=-1)

        # Value head
        v = jax.nn.relu(nn.Dense(128)(global_feat))
        v = jax.nn.relu(nn.Dense(64)(v))
        value = nn.Dense(1)(v).squeeze(-1)

        return action_logits, value

policy = GeneratorPolicyV2()
dummy_state = jnp.zeros((1, SEQ_LEN, 6))
dummy_meta = jnp.zeros((1, METADATA_DIM))
params = policy.init(jax.random.PRNGKey(0), dummy_state, dummy_meta)
num_params = sum(p.size for p in tree_util.tree_leaves(params))
print(f"GeneratorPolicyV2 initialized. {num_params:,} parameters (< 5M budget).")

In [ ]:
def sample_trajectory(params, key, wt_seq, metadata, num_edits=NUM_EDITS, temperature=TEMPERATURE):
    seq = wt_seq.copy()
    mutated = jnp.zeros(SEQ_LEN, dtype=jnp.bool_)
    actions = []
    log_probs = []

    for step in range(num_edits):
        key, action_key = jax.random.split(key)
        mutated_ch = mutated.astype(jnp.float32)[:, None]
        state_6ch = jnp.concatenate([seq, mutated_ch], axis=-1)[None, ...]
        meta_batch = metadata[None, ...]

        raw_logits, _ = policy.apply(params, state_6ch, meta_batch)
        raw_logits = raw_logits[0]

        # SOTA Vectorized Masking: 1000x faster than Python loops
        m_exp = jnp.repeat(mutated, VOCAB_SIZE)
        mask = jnp.logical_not(jnp.concatenate([m_exp, jnp.array([True])]))
        masked_logits = jnp.where(mask, raw_logits, -1e9)
        scaled_logits = masked_logits / temperature

        action = jax.random.categorical(action_key, scaled_logits)
        lp = jax.nn.log_softmax(masked_logits)[action]

        actions.append(int(action))
        log_probs.append(float(lp))

        pos, base = int(action) // VOCAB_SIZE, int(action) % VOCAB_SIZE
        seq = seq.at[pos].set(jax.nn.one_hot(base, VOCAB_SIZE))
        mutated = mutated.at[pos].set(True)

    return seq, np.array(actions, dtype=np.int32), np.array(log_probs, dtype=np.float32)

def onehot_to_acgtn(seq_onehot):
    BASES = "ACGTN"
    return "".join(BASES[i] for i in np.argmax(np.array(seq_onehot), axis=-1))

In [ ]:
def generate_trajectories(params, num_trajectories=NUM_TRAJECTORIES):
    wt_seq = jax.nn.one_hot(jnp.zeros(SEQ_LEN, dtype=jnp.int32), VOCAB_SIZE)
    metadata = jnp.ones(METADATA_DIM)

    all_actions, all_logprobs, all_seqs = [], [], []
    key = jax.random.PRNGKey(42)
    t0 = time.time()

    print(f"--- Starting Generation: {num_trajectories} trajectories ---")
    import sys
    pbar = tqdm(range(num_trajectories), desc="Generating Trajectories")
    for i in pbar:
        key, traj_key = jax.random.split(key)
        terminal_seq, actions, log_probs = sample_trajectory(params, traj_key, wt_seq, metadata)
        all_actions.append(actions)
        all_logprobs.append(log_probs)
        all_seqs.append(onehot_to_acgtn(terminal_seq))

        if i % max(1, num_trajectories // 20) == 0 and i > 0:
            rate = (i + 1) / (time.time() - t0)
            pbar.set_postfix(rate=f"{rate:.1f} traj/s")
            print(f"  > Progress: {i+1}/{num_trajectories} | Rate: {rate:.1f} traj/s")
            sys.stdout.flush()

    elapsed = time.time() - t0
    pbar.close()
    print(f"Generated {num_trajectories} trajectories in {elapsed:.1f}s")

    np.savez_compressed(TRAJ_PATH,
        actions=np.stack(all_actions),
        forward_log_probs=np.stack(all_logprobs),
        sequences=np.array(all_seqs, dtype=object),
        seq_len=SEQ_LEN, num_edits=NUM_EDITS, temperature=TEMPERATURE,
    )
    size_mb = os.path.getsize(TRAJ_PATH) / 1024 / 1024
    print(f"Saved: {TRAJ_PATH} ({size_mb:.1f} MB)")
    return all_seqs, all_actions, all_logprobs

all_seqs, all_actions, all_logprobs = generate_trajectories(params)
